# 03 — Designing ClaimClerk's extraction prompt

This notebook walks through three iterations of the JSON-extraction
prompt for ClaimClerk, working against Sarah Chen's appeal email
(the lifecycle anchor — artifact A1, landed by the setup notebook c0001). The final iteration
is registered to MLflow Prompt Registry as artifact A5, ready for
the ClaimClerk chain to load by name@alias.

**Model**: `databricks-meta-llama-3-1-8b-instruct` (pay-per-token).
Llama 3.1 8B is the right size for prompt iteration: fast enough that
each iteration runs in under two seconds, capable enough that real
prompt-engineering effects are visible. Production ClaimClerk graduates to `databricks-meta-llama-3-3-70b-instruct` on
provisioned throughput (Llama 3.1 70B was retired from the FM API;
the 3.3 70B endpoint is the current PT-capable 70B option).

**Prerequisites**: the setup notebook c0001 ran successfully — your
`${catalog}.pawshield.bootstrap` Volume holds the canonical emails
and `${catalog}.pawshield.customers` resolves Sarah Chen.

In [0]:
# Pin mlflow<3.13 — 3.13.0 has a regression where eval_item.trace is None
# during mlflow.genai.evaluate (trace not associated with eval_request_id).
%pip install -q --force-reinstall databricks-openai "mlflow[databricks]>=3.12,<3.13" "databricks-vectorsearch<0.74"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.13.0 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "genaicert")
dbutils.widgets.text("llm_endpoint", "databricks-meta-llama-3-1-8b-instruct")
catalog = dbutils.widgets.get("catalog")
llm_endpoint = dbutils.widgets.get("llm_endpoint")
print(f"catalog:      {catalog}")
print(f"llm_endpoint: {llm_endpoint}")

catalog:      genaicert
llm_endpoint: databricks-meta-llama-3-1-8b-instruct


## 1. Read Sarah Chen's email (the canonical extraction target)

In [0]:
email_path = f"/Volumes/{catalog}/pawshield/bootstrap/claim_emails/sarah_chen_20260328_098473.eml"
print(f"email_path: {email_path}")

email_path: /Volumes/genaicert/pawshield/bootstrap/claim_emails/sarah_chen_20260328_098473.eml


In [0]:
with open(email_path, "r") as f:
    sarah_email = f.read()
print(sarah_email)

From: sarah.chen.austin@gmail.com
To: claims@pawshield.com
Date: Sat, 28 Mar 2026 09:47:14 -0500
Subject: Claim CLM-2026-03-00471 =?utf-8?b?4oCU?= Help please??
Content-Type: text/plain; charset="utf-8"
Content-Transfer-Encoding: 8bit
MIME-Version: 1.0

Hi,

I submitted Biscuit's ER claim two weeks ago and only got $512 back
on a $890 bill. I'm being told it's because of my deductible but I
already paid that this year. This is really frustrating, my cat is
sick and I'm out of pocket. Can someone please call me — 512-555-0188.

Thanks,
Sarah



## 2. The model client

The OpenAI-compatible client works against any Databricks serving
endpoint. `DatabricksOpenAI()` returns a pre-authenticated client
wired to the workspace's `serving-endpoints` URL — no API key
needed; Databricks handles auth through the notebook session.

(The older `WorkspaceClient().serving_endpoints.get_open_ai_client()`
path was officially deprecated in `databricks-sdk-py` v0.88.0
(2026-02-12) in favour of the dedicated `databricks-openai` and
`databricks-langchain` packages — it still functions on current SDK
versions but emits a deprecation warning. The book uses
`DatabricksOpenAI()` directly going forward.)

In [0]:
# Source: https://api-docs.databricks.com/python/databricks-ai-bridge/latest/databricks_openai.html
import time

from databricks_openai import DatabricksOpenAI
from openai import RateLimitError

client = DatabricksOpenAI()

# Pay-per-token FM endpoints have a per-workspace QPS cap. The bulk-sample
# loop below fires ~20 calls back-to-back, which is enough to trip
# `REQUEST_LIMIT_EXCEEDED` on a shared workspace. Two cheap defences keep
# the teaching notebook running without graduating to provisioned throughput:
#   * a 0.3 s post-call sleep keeps us under the QPS cap by construction
#   * exponential-backoff retry on 429 recovers from the occasional burst
# Production at PolicyPal scale graduates to provisioned throughput.
THROTTLE_SECONDS = 0.3
MAX_RETRIES = 4


def ask(system: str, user: str, *, temperature: float = 0.0) -> str:
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=llm_endpoint,
                messages=[{"role": "system", "content": system},
                          {"role": "user",   "content": user}],
                temperature=temperature,
                max_tokens=400,
            )
            time.sleep(THROTTLE_SECONDS)
            return resp.choices[0].message.content
        except RateLimitError:
            if attempt == MAX_RETRIES - 1:
                raise
            backoff = 2 ** (attempt + 1)
            print(f"  rate-limited; sleeping {backoff}s then retrying...")
            time.sleep(backoff)

## 3. Iteration v1 — naive zero-shot

"Just ask for JSON" — what most people try first.

In [0]:
PROMPT_V1 = "Extract the claim information from this email as JSON."
print(PROMPT_V1)

Extract the claim information from this email as JSON.


In [0]:
out_v1 = ask(PROMPT_V1, sarah_email)
print(out_v1)

Here is the claim information extracted from the email as JSON:

```
{
  "claimNumber": "CLM-2026-03-00471",
  "claimType": "ER",
  "claimAmount": 890,
  "reimbursedAmount": 512,
  "deductibleStatus": "paid",
  "additionalInfo": "Cat is sick, out of pocket"
}
```

Note that the "deductibleStatus" field is not explicitly stated in the email, but based on the context, it can be inferred that the deductible has been paid.


The output is JSON-ish but the schema drifts — keys vary
(`claim_number` vs `claim_id`), commentary may wrap the JSON,
and required fields like `intent` and `sentiment` are absent.
We need to be explicit about all four parts of a production prompt.

## 4. Iteration v2 — four-part prompt (role + task + constraints + example)

In [0]:
SYSTEM_V2 = """You are an email-parsing assistant for a pet insurance claims team.

TASK
Extract structured information from a customer email.

CONSTRAINTS
Output a single JSON object with exactly these keys:
- claim_id        (string, format CLM-YYYY-MM-NNNNN, or null)
- customer_id     (string, format CUST-NAME-NNN, or null)
- pet_name        (string, or null)
- contact_phone   (string in hyphenated format like 512-555-0188, or null)
- intent          (one of: claim_status, policy_q, appeal, complaint,
                          vet_lookup, emergency, other)
- sentiment       (one of: positive, neutral, frustrated, angry, panicked)
- urgency         (one of: low, medium, high, emergency)

Do not include any text before or after the JSON. Do not add fields.
Use null for any field the email does not mention.

EXAMPLE
INPUT:
"From: alex.tran@gmail.com
Subject: Cancel my account

Please cancel my PawShield policy. Thanks, Alex."

OUTPUT:
{"claim_id": null, "customer_id": null, "pet_name": null,
 "contact_phone": null, "intent": "other", "sentiment": "neutral",
 "urgency": "low"}
"""
print(f"SYSTEM_V2 is {len(SYSTEM_V2)} chars")

SYSTEM_V2 is 1078 chars


In [0]:
out_v2 = ask(SYSTEM_V2, sarah_email)
print(out_v2)

{"claim_id": "CLM-2026-03-00471", "customer_id": "CUST-SARAH-CHEN", "pet_name": "Biscuit", "contact_phone": "512-555-0188", "intent": "claim_status", "sentiment": "frustrated", "urgency": "high"}


v2 is structurally correct (all required keys, no extra fields,
valid JSON) but watch the `contact_phone` field — on some runs
the model emits `512.555.0188` (period-separated) instead of
the hyphen format the schema requires. The written constraint
("hyphenated format like 512-555-0188") is not enough; the model
needs few-shot examples that show the exact format.

## 5. Iteration v3 — add few-shot examples for format stability

In [0]:
SYSTEM_V3 = SYSTEM_V2 + """
ADDITIONAL EXAMPLES

INPUT:
"From: devon@protonmail.com
Subject: Status on claim CLM-2025-12-04881

Hello, where is my December claim for my dog Bruno? Phone (415) 555 8821."

OUTPUT:
{"claim_id": "CLM-2025-12-04881", "customer_id": null, "pet_name": "Bruno",
 "contact_phone": "415-555-8821", "intent": "claim_status",
 "sentiment": "neutral", "urgency": "low"}

INPUT:
"My dog just ate chocolate!! Please help, this is urgent.
Call 206.555.0732 — Min-jun"

OUTPUT:
{"claim_id": null, "customer_id": null, "pet_name": null,
 "contact_phone": "206-555-0732", "intent": "emergency",
 "sentiment": "panicked", "urgency": "emergency"}
"""
print(f"SYSTEM_V3 is {len(SYSTEM_V3)} chars")

SYSTEM_V3 is 1711 chars


In [0]:
out_v3 = ask(SYSTEM_V3, sarah_email)
print(out_v3)

{"claim_id": "CLM-2026-03-00471", "customer_id": null, "pet_name": "Biscuit",
 "contact_phone": "512-555-0188", "intent": "claim_status", "sentiment": "frustrated",
 "urgency": "medium"}


v3 should now reliably produce `512-555-0188` (the hyphenated
format). The few-shot examples encode the format more reliably
than the written rule did.

## 6. Apply v3 to the full canonical set

Six canonical emails, one per recurring family. If schema
satisfaction holds across all six, the prompt is ready for the
bulk-sample test.

In [0]:
import json

CANONICAL = [
    ("sarah_chen_20260328_098473.eml",        "F1 Chen — appeal"),
    ("rodriguez_carmen_20260205_037119.eml",  "F2 Rodríguez — Spanish policy q"),
    ("whitfield_eleanor_20260311_062804.eml", "F3 Whitfield — audit request"),
    ("park_minjun_20260411_104822.eml",       "F4 Park — chocolate emergency"),
    ("johnson_tasha_20260319_088201.eml",     "F5 Johnson — billing pattern"),
    ("brennan_conor_20260408_119522.eml",     "F6 Brennan — wellness rider"),
]
print(f"{len(CANONICAL)} canonical emails")

6 canonical emails


In [0]:
results = []
for filename, label in CANONICAL:
    with open(f"/Volumes/{catalog}/pawshield/bootstrap/claim_emails/{filename}") as f:
        body = f.read()
    raw = ask(SYSTEM_V3, body)
    try:
        parsed = json.loads(raw)
        keys_ok = set(parsed.keys()) == {
            "claim_id", "customer_id", "pet_name", "contact_phone",
            "intent", "sentiment", "urgency",
        }
        status = "OK" if keys_ok else "SCHEMA-DRIFT"
    except json.JSONDecodeError as e:
        parsed = None
        status = f"NOT-JSON ({e.msg})"
    results.append((label, status, parsed))
    print(f"  {label:45s}  {status}")

  F1 Chen — appeal                               OK
  F2 Rodríguez — Spanish policy q                OK
  F3 Whitfield — audit request                   OK
  F4 Park — chocolate emergency                  OK
  F5 Johnson — billing pattern                   OK
  F6 Brennan — wellness rider                    OK


Inspect the parsed output for Sarah's case — this is the canonical extraction result for the lifecycle thread.

In [0]:
print(json.dumps(results[0][2], indent=2))

{
  "claim_id": "CLM-2026-03-00471",
  "customer_id": null,
  "pet_name": "Biscuit",
  "contact_phone": "512-555-0188",
  "intent": "claim_status",
  "sentiment": "frustrated",
  "urgency": "medium"
}


## 7. Lock the prompt as artifact A5

Once the prompt passes the canonical set (and a bulk sample —
see optional cell below), the exact text becomes A5. Two
registration patterns are common; pick one based on your
workspace's MLflow version.

In [0]:
# Pattern A — MLflow Prompt Registry (MLflow 3+; canonical going forward).
# This registers the prompt as a first-class versioned asset that the
# ClaimClerk chain references by name@alias.
#
# Source: https://mlflow.org/docs/latest/api_reference/python_api/mlflow.genai.html
# Current API (verified May 2026 against the MLflow GenAI Python API docs):
#   - mlflow.genai.register_prompt(name=..., template=..., commit_message=...)
#   - mlflow.genai.set_prompt_alias(name=..., alias=..., version=...)
#   - mlflow.genai.load_prompt("prompts:/<name>@<alias>")

try:
    import mlflow
    registered = mlflow.genai.register_prompt(
        name=f"{catalog}.pawshield.claimclerk_extraction",
        template=SYSTEM_V3,
        commit_message="v1 — passes canonical six-family set",
    )
    # Tag the freshly-registered version as @champion so the ClaimClerk chain
    # (which loads `prompts:/.../claimclerk_extraction@champion`) resolves.
    mlflow.genai.set_prompt_alias(
        name=f"{catalog}.pawshield.claimclerk_extraction",
        alias="champion",
        version=registered.version,
    )
    print(
        f"Registered claimclerk_extraction v{registered.version} via "
        f"MLflow Prompt Registry and tagged @champion."
    )
except (AttributeError, ImportError):
    # Pattern B — MLflow log_artifact (works on any MLflow version).
    # Less rich (no native versioning, no alias support) but always available.
    import mlflow
    with mlflow.start_run(run_name="claimclerk_extraction_v1"):
        mlflow.log_text(SYSTEM_V3, "claimclerk_extraction.txt")
    print("Logged as text artifact (MLflow Prompt Registry not available).")

Registered claimclerk_extraction v11 via MLflow Prompt Registry and tagged @champion.


In [0]:
mlflow.genai.load_prompt(f"prompts:/{catalog}.pawshield.claimclerk_extraction@champion")

PromptVersion(name=genaicert.pawshield.claimclerk_extraction, version=11, template=You are an email-parsing assis...)

## 8. (Optional) Bulk-sample test

The canonical set covers the hard cases by construction. The
long tail of bulk-generated emails covers the typical cases.
Run this cell once before declaring v1 ready for chain wiring.

In [0]:
import os, random

bulk_dir = f"/Volumes/{catalog}/pawshield/bootstrap/claim_emails"
random.seed(42)
all_files = [f for f in os.listdir(bulk_dir) if f.endswith(".eml")]
canon_names = {c[0] for c in CANONICAL}
bulk_sample = random.sample(
    [f for f in all_files if f not in canon_names],
    k=20,
)
print(f"bulk_sample: {len(bulk_sample)} emails (excluding {len(canon_names)} canonical)")

bulk_sample: 20 emails (excluding 6 canonical)


In [0]:
bulk_ok, bulk_drift = 0, 0
for filename in bulk_sample:
    with open(f"{bulk_dir}/{filename}") as f:
        body = f.read()
    raw = ask(SYSTEM_V3, body)
    try:
        parsed = json.loads(raw)
        ok = set(parsed.keys()) == {"claim_id", "customer_id", "pet_name",
                                     "contact_phone", "intent", "sentiment",
                                     "urgency"}
        if ok: bulk_ok += 1
        else:  bulk_drift += 1
    except json.JSONDecodeError:
        bulk_drift += 1

print(f"Bulk sample: {bulk_ok}/{len(bulk_sample)} pass schema, "
      f"{bulk_drift} drift or non-JSON")
print(f"Ship gate: >= 95% pass rate on bulk + 100% on canonical.")

Bulk sample: 20/20 pass schema, 0 drift or non-JSON
Ship gate: >= 95% pass rate on bulk + 100% on canonical.


## Soft → hard contract — promote to `responseFormat`

The v3 prompt produces JSON because we told it to in prose. When
the JSON shape becomes load-bearing for a downstream consumer
(a serving PyFunc, a batch SQL pipeline), the disciplined move
is to promote the soft prose constraint to a hard decoder
constraint via `ai_query`'s `responseFormat`. The model can no
longer drift the shape; either the row returns the typed struct
or `failOnError=>false` lands the row with an `errorMessage`.
Source: docs.databricks.com/aws/en/sql/language-manual/functions/ai_query

In [0]:
# Demonstrate the soft→hard upgrade. The JSON-schema shape matches the
# four-part prompt's documented output schema; the constraint moves
# from prose into the decoder. Requires Runtime 15.4 LTS+ for chat-FM
# `responseFormat` support. The DDL-STRUCT shorthand also appears in
# the ai_query docs but is not accepted on all runtimes — JSON Schema
# is the safer, more portable form for structured-output enforcement.
# Source: docs.databricks.com/aws/en/sql/language-manual/functions/ai_query
spark.sql("""
    SELECT
        ai_query(
            endpoint       => 'databricks-meta-llama-3-1-8b-instruct',
            request        => CONCAT(
                'Extract claim fields as JSON. ',
                'I submitted Biscuit ER claim CLM-2026-03-00471 '
                'two weeks ago and only got $512 back. Sarah Chen, '
                'customer CUST-CHEN-001, phone 512-555-0188.'
            ),
            responseFormat => '{
                "type": "json_schema",
                "json_schema": {
                    "name": "claim_extraction",
                    "schema": {
                        "type": "object",
                        "properties": {
                            "claim_id":      {"type": ["string", "null"]},
                            "customer_id":   {"type": ["string", "null"]},
                            "pet_name":      {"type": ["string", "null"]},
                            "contact_phone": {"type": ["string", "null"]},
                            "intent":        {"type": "string", "enum": ["claim_status", "policy_q", "appeal", "complaint", "vet_lookup", "emergency", "other"]},
                            "sentiment":     {"type": "string", "enum": ["positive", "neutral", "frustrated", "angry", "panicked"]},
                            "urgency":       {"type": "string", "enum": ["low", "medium", "high", "emergency"]}
                        },
                        "required": ["claim_id", "customer_id", "pet_name", "contact_phone", "intent", "sentiment", "urgency"],
                        "additionalProperties": false
                    },
                    "strict": true
                }
            }',
            failOnError    => false
        ) AS extracted
""").show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|extracted                                                                                                                                                                                               |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{{"customer_id": "CUST-CHEN-001", "intent": "claim_status", "sentiment": "frustrated", "claim_id": "CLM-2026-03-00471", "urgency": "low", "contact_phone": "512-555-0188", "pet_name": "Biscuit"}, NULL}|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## What's next

The prompt is locked. The ingestion side (notebook c0401) extracts
the policy PDFs in the Volume into a Delta landing table; c0501 then
chunks them for retrieval. The extraction prompt sits in MLflow until
the ClaimClerk chain reads it back via `claimclerk_extraction@champion`.